In [ ]:
# Necessary libraries
import numpy as np
import marimo as mo
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Sklearn preprocessing libraries
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV,
)

# Regression libraries
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import VotingRegressor, StackingRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# fg-data-profiling
from data_profiling import ProfileReport

# Metrices
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Ignore warnings
import warnings

warnings.filterwarnings("ignore")

# Logging
import logging
from pathlib import Path

In [ ]:
np.random.seed(42)

In [ ]:
dataset = pd.read_csv("data/raw/instagram_usage_lifestyle.csv")

In [ ]:
df = dataset.sample(frac=0.00001, random_state=42).reset_index(drop=True)

In [ ]:
df

In [ ]:
# profile = ProfileReport(
#     df=dataset,
#     title="Effect of Social Media (Instagram) in Human Happiness",
#     explorative=True,
# )

# profile.to_file("ydata.html")

In [ ]:
mo.md(r"""
**Decisions based on y-data profiling:**

- Have to drop *user_id, app__name, last_login_date* features.
- Have to impute the missing values in the *perceived_stress_score, hobbies_count, social_events_per_month, travel_frequency_per_year, posts_created_per_week, ads_clicked_per_day, linked_accounts_count* features.
- Have to convert the categorical values into numerical values
""")

<span class="markdown prose dark:prose-invert contents"><span class="paragraph"><strong>Decisions based on y-data profiling:</strong></span>
<ul>
<li>Have to drop <em>user_id, app__name, last_login_date</em> features.</li>
<li>Have to impute the missing values in the <em>perceived_stress_score, hobbies_count, social_events_per_month, travel_frequency_per_year, posts_created_per_week, ads_clicked_per_day, linked_accounts_count</em> features.</li>
<li>Have to convert the categorical values into numerical values</li>
</ul></span>

In [ ]:
# Drop the necessary features

if "user_id" in df.columns and "app_name" in df.columns:
    df.drop(columns=["user_id", "app_name"], inplace=True)

In [ ]:
if "last_login_date" in df.columns:
    df.drop(columns=["last_login_date"], inplace=True)

In [ ]:
if (
    "user_id" not in df.columns
    and "app_name" not in df.columns
    and "last_login_date" not in df.columns
):
    print("Dropping done!")
else:
    print("Error occurred while dropping!")

In [ ]:
# for col in df.columns:
#     print(col, df[col].dtype)

In [ ]:
numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()
categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

In [ ]:
mo.hstack(
    [
        numerical_features,
        categorical_features,
    ]
)

In [ ]:
nominal_features = [
    "gender",
    "country",
    "urban_rural",
    "employment_status",
    "relationship_status",
    "has_children",
    "smoking",
    "alcohol_frequency",
    "uses_premium_features",
    "content_type_preference",
    "preferred_content_theme",
    "privacy_setting_level",
    "two_factor_auth_enabled",
    "biometric_login_used",
    "subscription_status",
]

ordinal_features = ["income_level", "education_level", "diet_quality"]

In [ ]:
target_feature = "self_reported_happiness"

In [ ]:
mo.md(r"""
**Imputing the missing values for both numerical and categorical features**
""")

<span class="markdown prose dark:prose-invert contents"><span class="paragraph"><strong>Imputing the missing values for both numerical and categorical features</strong></span></span>

In [ ]:
# For numerical features

numerical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

In [ ]:
# For nominal categorical features (constant -> most frequent)

nominal_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
        ),
    ]
)

In [ ]:
# For ordinal categorical features (constant -> most frequent)

ordinal_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OrdinalEncoder(
                categories=[
                    ["Low", "Lower-middle", "Upper-middle", "Middle", "High"],
                    [
                        "High school",
                        "Some college",
                        "Bachelor’s",
                        "Master’s",
                        "Other",
                    ],
                    ["Very poor", "Poor", "Average", "Good", "Excellent"],
                ],
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            ),
        ),
    ]
)

In [ ]:
# Combine all pipelines

preprocessor = ColumnTransformer(
    [
        ("numerical", numerical_pipeline, numerical_features),
        ("nominal", nominal_pipeline, nominal_features),
        ("ordinal", ordinal_pipeline, ordinal_features),
    ]
)

In [ ]:
X = df.drop(target_feature, axis=1)
y = df[target_feature]

In [ ]:
# Train and test split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

In [ ]:
# Base learners

lr_reg = LinearRegression()
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
gb_reg = GradientBoostingRegressor(n_estimators=100, random_state=42)

In [ ]:
# Voting regressor

voting_reg = VotingRegressor(
    estimators=[
        ("lr_reg", lr_reg),
        ("rf_reg", rf_reg),
        ("gb_reg", gb_reg),
    ]
)

In [ ]:
# Stacking regressor

stacking_reg = StackingRegressor(
    estimators=[
        ("lr_reg", lr_reg),
        ("rf_reg", rf_reg),
        ("gb_reg", gb_reg),
    ],
    final_estimator=Ridge(),  # meta learner
)

In [ ]:
mo.md(r"""
**Model Training**
""")

<span class="markdown prose dark:prose-invert contents"><span class="paragraph"><strong>Model Training</strong></span></span>

In [ ]:
# Dictionary of all models

model_to_train = {
    "Linear Regression": lr_reg,
    "Random Forest Regression": rf_reg,
    "Gradient Boosting Regression": gb_reg,
    "Voting Ensemble": voting_reg,
    "Stacking Ensemble": stacking_reg,
}

In [ ]:
mo.md(r"""
**Model Training and Evaluation**
""")

<span class="markdown prose dark:prose-invert contents"><span class="paragraph"><strong>Model Training and Evaluation</strong></span></span>

In [ ]:
try:
    logger.info("Starting baseline model training (without PCA)...")
    results = []

    for _name, _model in model_to_train.items():
        try:
            logger.info(f"Training {_name} without PCA...")

            # Full pipeline with model

            full_pipeline = Pipeline(
                steps=[("preprocessor", preprocessor), ("model", _model)]
            )

            # Training
            full_pipeline.fit(X_train, y_train)

            # Prediction
            _y_pred = full_pipeline.predict(X_test)

            # Evaluation

            _r2 = r2_score(y_test, _y_pred)
            _mae = mean_absolute_error(y_test, _y_pred)
            _mse = mean_squared_error(y_test, _y_pred)
            _rmse = np.sqrt(_mse)

            results.append(
                {"Model": _name, "R2 Score": _r2, "MAE": _mae, "RMSE": _rmse}
            )

        except Exception as e:
            logger.error(
                f"Error training {_name} with PCA: {str(e)}", exc_info=True
            )
            continue

    results_df = pd.DataFrame(results).sort_values("R2 Score", ascending=False)

except Exception as e:
    logger.error(f"Critical error in PCA training: {str(e)}", exc_info=True)

In [ ]:
# Creating logs directory
log_dir = Path("logs")
log_dir.mkdir(exist_ok=True)

log_file = log_dir / "crimson_nebula.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[logging.FileHandler(log_file), logging.StreamHandler()],
    force=True,
)

logger = logging.getLogger("crimson-nebula")
logger.info("Logging is set up.")
logging.info("crimson-nebula - Model Training Started.")

In [ ]:
results_df

In [ ]:
mo.md(r"""
**Principal Component Analysis**
""")

<span class="markdown prose dark:prose-invert contents"><span class="paragraph"><strong>Principal Component Analysis</strong></span></span>

In [ ]:
# Transform the data using preprocessor

X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

In [ ]:
# Apply principal component analysis

pca = PCA()

X_train_pca = pca.fit_transform(X_train_transformed)
X_test_pca = pca.transform(X_test_transformed)

In [ ]:
# Explained variance ratio

explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

In [ ]:
# Plot explained variance
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Individual explained variance
ax1.bar(range(1, len(explained_variance) + 1), explained_variance)
ax1.set_xlabel("Principal Component")
ax1.set_ylabel("Explained Variance Ratio")
ax1.set_title("PCA - Explained Variance by Component")

# Cumulative explained variance
ax2.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, "bo-")
ax2.axhline(y=0.95, color="r", linestyle="--", label="95% Variance")
ax2.set_xlabel("Number of Components")
ax2.set_ylabel("Cumulative Explained Variance")
ax2.set_title("PCA - Cumulative Explained Variance")
ax2.legend()
ax2.grid(True)

plt.tight_layout()

In [ ]:
# Find number of components for 95% variance
n_components_95 = np.argmax(cumulative_variance >= 0.95) + 1

mo.md(f"**Number of components needed for 95% variance: {n_components_95}**")

In [ ]:
# PCA with optimal number of components
pca_optimal = PCA(n_components=n_components_95)

In [ ]:
try:
    logger.info(
        f"Starting PCA-based model training with {n_components_95} components..."
    )

    results_with_pca = []

    for _name, _model in model_to_train.items():
        try:
            logger.info(f"Training {_name} with PCA...")
            # Pipeline with PCA
            full_pipeline_pca = Pipeline(
                steps=[
                    ("preprocessor", preprocessor),
                    ("pca", PCA(n_components=n_components_95)),
                    ("model", _model),
                ]
            )

            # Training
            full_pipeline_pca.fit(X_train, y_train)

            # Prediction
            _y_pred = full_pipeline_pca.predict(X_test)

            # Evaluation
            _r2 = r2_score(y_test, _y_pred)
            _mae = mean_absolute_error(y_test, _y_pred)
            _mse = mean_squared_error(y_test, _y_pred)
            _rmse = np.sqrt(_mse)

            logger.info(
                f"{_name} with PCA - R2: {_r2:.4f}, MAE: {_mae:.4f}, RMSE: {_rmse:.4f}"
            )

            results_with_pca.append(
                {"Model": _name, "R2 Score": _r2, "MAE": _mae, "RMSE": _rmse}
            )
        except Exception as e:
            logger.error(
                f"Error training {_name} with PCA: {str(e)}", exc_info=True
            )
            continue

    results_with_pca_df = pd.DataFrame(results_with_pca).sort_values(
        "R2 Score", ascending=False
    )

except Exception as e:
    logger.critical(f"Critical error in PCA training: {str(e)}", exc_info=True)
    raise

In [ ]:
results_with_pca_df

In [ ]:
# Hyperparameter grids for each _model

param_grids = {
    "Random Forest Regression": {
        "model__n_estimators": [50, 100, 200],
        "model__max_depth": [10, 20, 30, None],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4],
    },
    "Gradient Boosting Regression": {
        "model__n_estimators": [50, 100, 200],
        "model__learning_rate": [0.01, 0.1, 0.2],
        "model__max_depth": [3, 5, 7],
        "model__subsample": [0.8, 0.9, 1.0],
    },
    "Linear Regression": {},  # No hyperparameters to tune
    "Voting Ensemble": {
        "model__rf_reg__n_estimators": [50, 100],
        "model__gb_reg__n_estimators": [50, 100],
        "model__gb_reg__learning_rate": [0.01, 0.1],
    },
    "Stacking Ensemble": {
        "model__rf_reg__n_estimators": [50, 100],
        "model__gb_reg__n_estimators": [50, 100],
        "model__final_estimator__alpha": [0.1, 1.0, 10.0],
    },
}

In [ ]:
# For RandomizedSearchCV - larger search space

param_distributions = {
    "Random Forest Regression": {
        "model__n_estimators": [50, 100, 150, 200, 300],
        "model__max_depth": [5, 10, 15, 20, 25, 30, None],
        "model__min_samples_split": [2, 5, 10, 15],
        "model__min_samples_leaf": [1, 2, 4, 6],
        "model__max_features": ["sqrt", "log2", None],
    },
    "Gradient Boosting Regression": {
        "model__n_estimators": [50, 100, 150, 200, 300],
        "model__learning_rate": [0.01, 0.05, 0.1, 0.15, 0.2],
        "model__max_depth": [3, 4, 5, 6, 7, 8],
        "model__subsample": [0.7, 0.8, 0.9, 1.0],
        "model__min_samples_split": [2, 5, 10],
    },
}

In [ ]:
try:
    logger.info("Starting GridSearchCV hyperparameter tuning...")

    grid_search_results = []
    best_models_grid = {}

    for _name, _model in model_to_train.items():
        if _name in param_grids and param_grids[_name]:
            try:
                logging.info(f"\nTunning {_name} with GridSearchCV...")

                # Create pipeline
                _pipeline = Pipeline(
                    [("preprocessor", preprocessor), ("model", _model)]
                )
                grid_search = GridSearchCV(
                    _pipeline,
                    param_grids[_name],
                    cv=5,
                    scoring="r2",
                    n_jobs=-1,
                    verbose=1,
                )

                # Fit
                grid_search.fit(X_train, y_train)

                # Best _model
                best_models_grid[_name] = grid_search.best_estimator_

                # Predict
                _y_pred = grid_search.predict(X_test)

                # Evaluate
                _r2 = r2_score(y_test, _y_pred)
                _mae = mean_absolute_error(y_test, _y_pred)
                _mse = mean_squared_error(y_test, _y_pred)
                _rmse = np.sqrt(_mse)

                grid_search_results.append(
                    {
                        "_model": _name,
                        "Best Params": str(grid_search.best_params_),
                        "R2 Score": _r2,
                        "MAE": _mae,
                        "RMSE": _rmse,
                    }
                )

            except Exception as e:
                logger.error(
                    f"Error in GridSearchCV for {_name}: {str(e)}",
                    exc_info=True,
                )
                continue
        else:
            logger.info(
                f"Skipping GridSearchCV for {_name} - no hyperparameters to tune"
            )

    grid_results_df = pd.DataFrame(grid_search_results).sort_values(
        "R2 Score", ascending=False
    )
    logger.info("GridSearchCV tuning completed successfully")

except Exception as e:
    logger.critical(f"Critical error in GridSearchCV: {str(e)}", exc_info=True)
    raise

In [ ]:
grid_results_df

In [ ]:
random_search_results = []
best_models_random = {}

try:
    logger.info("Starting RandomizedSearchCV hyperparameter tuning...")

    for _name, _model in model_to_train.items():
        if _name in param_distributions:
            try:
                logging.info(f"\nTunning {_name} with RandomizedSearchCV...")

                # Create pipeline
                _pipeline = Pipeline(
                    [("preprocessor", preprocessor), ("model", _model)]
                )
                random_search = RandomizedSearchCV(
                    _pipeline,
                    param_distributions[_name],
                    n_iter=20,
                    cv=5,
                    scoring="r2",
                    n_jobs=-1,
                    verbose=1,
                    random_state=42,
                )

                # Fit
                random_search.fit(X_train, y_train)

                # Best model
                best_models_random[_name] = random_search.best_estimator_

                # Predict
                _y_pred = random_search.predict(X_test)

                # Evaluate
                _r2 = r2_score(y_test, _y_pred)
                _mae = mean_absolute_error(y_test, _y_pred)
                _mse = mean_squared_error(y_test, _y_pred)
                _rmse = np.sqrt(_mse)

                random_search_results.append(
                    {
                        "Model": _name,
                        "Best Params": str(random_search.best_params_),
                        "R2 Score": _r2,
                        "MAE": _mae,
                        "RMSE": _rmse,
                    }
                )
            except Exception as e:
                logger.error(
                    f"Error in RandomizedSearchCV for {_name}: {str(e)}",
                    exc_info=True,
                )
                continue
        else:
            print(f"Skipping {_name} - no hyperparameters to tune")

    random_results_df = pd.DataFrame(random_search_results).sort_values(
        "R2 Score", ascending=False
    )
    logger.info("RandomizedSearchCV tuning completed successfully")
except Exception as e:
    logger.critical(
        f"Critical error in RandomizedSearchCV: {str(e)}", exc_info=True
    )
    raise

In [ ]:
random_results_df

In [ ]:
mo.md(r"""
**Choose best model**
""")

<span class="markdown prose dark:prose-invert contents"><span class="paragraph"><strong>Choose best model</strong></span></span>

In [ ]:
grid_results_df["Method"] = "GridSearchCV"
random_results_df["Method"] = "RandomizedSearchCV"
results_with_pca_df["Method"] = "PCA"

In [ ]:
# Combine all results
all_results = pd.concat(
    [grid_results_df, random_results_df, results_with_pca_df],
    ignore_index=True,
)

In [ ]:
all_results_sorted = all_results.sort_values("R2 Score", ascending=False)

In [ ]:
best_model_info = all_results_sorted.iloc[0]

In [ ]:
best_model_info

In [ ]:
mo.md(r"""
**Save the model**
""")

<span class="markdown prose dark:prose-invert contents"><span class="paragraph"><strong>Save the model</strong></span></span>

In [ ]:
best_model_name = best_model_info["Model"]
best_method = best_model_info["Method"]

if best_method == "GridSearchCV":
    best_model = best_models_grid[best_model_name]
elif best_method == "RandomizedSearchCV":
    best_model = best_models_random[best_model_name]
else:
    best_model = model_to_train[best_model_name]

In [ ]:
import pickle
import json

try:
    # Create models directory
    models_dir = Path("src/models")
    models_dir.mkdir(parents=True, exist_ok=True)

    # Save the model
    model_path = models_dir / "best_model.pkl"
    with open(model_path, "wb") as f:
        pickle.dump(best_model, f)

    logger.info(f"Model saved successfully at {model_path}")

except Exception as e:
    logger.error(f"Error saving model: {str(e)}", exc_info=True)

In [ ]:
try:
    logger.info("Retraining best model on full dataset...")

    # Prepare full dataset
    X_full = dataset.drop(target_feature, axis=1)
    y_full = dataset[target_feature]

    # Extract the model based on the method used
    if best_method in ["GridSearchCV", "RandomizedSearchCV"]:
        # These are Pipeline objects
        model_only = best_model.named_steps["model"]
    else:
        # PCA method returns raw model
        model_only = best_model

    # Create full pipeline with preprocessor and model
    full_pipeline_retrain = Pipeline(
        steps=[("preprocessor", preprocessor), ("model", model_only)]
    )

    # Retrain the model on full dataset
    full_pipeline_retrain.fit(X_full, y_full)

    # Update best_model to be the full pipeline
    best_model_r = full_pipeline_retrain

    logger.info(
        "Model retrained successfully on full dataset with preprocessor"
    )

except Exception as e:
    logger.error(f"Error retraining model: {str(e)}", exc_info=True)

In [ ]:
try:
    models_dir_r = Path("src/models")
    models_dir_r.mkdir(parents=True, exist_ok=True)

    # Save the retrained model
    model_path_r = models_dir_r / "crimson_nebula.pkl"
    with open(model_path_r, "wb") as file:
        pickle.dump(best_model_r, file)

    logger.info(f"Retrained model saved at {model_path_r}")

except Exception as e:
    logger.error(f"Error saving retrained model: {str(e)}", exc_info=True)